---
title: "Run-Length Encoding"
language: "Python"
difficulty: "Beginner"
topic: "String basics, Data structures"
slug: "run-length-encoding"
---

## Problem Statement

Implement two functions that together form a basic **run-length encoding (RLE)** codec.

**`rle_encode(s: str) -> str`**
Compress a string by replacing consecutive runs of the same character with the character followed by the count. `"aaabbc"` → `"a3b2c1"`.

**`rle_decode(s: str) -> str`**
Reverse the process: given a valid encoded string like `"a3b2c1"`, return the original `"aaabbc"`.

**Requirements for `rle_encode`:**
- Every character in the output is followed by its count, even if the count is 1.
- An empty string encodes to `""`.
- Input may contain any printable ASCII characters including spaces and digits.

**Requirements for `rle_decode`:**
- The encoded string is always a sequence of `(character)(positive integer)` pairs — no other format will be passed in.
- An empty string decodes to `""`.
- The count for a character may be more than one digit (e.g. `"a12"` means 12 consecutive `'a'`s).
- Raise `ValueError` with the message `"Invalid encoded string"` if the input does not match the expected format (e.g. two consecutive letters without a count, or a leading digit).

```python
def rle_encode(s: str) -> str:
    # TODO: walk s, count consecutive identical characters, and build the output.
    pass


def rle_decode(s: str) -> str:
    # TODO: walk the encoded string in (char, digits) pairs and reconstruct the original.
    pass
```

## Expected Output / Test Cases

```python
# --- rle_encode ---

# Happy path
assert rle_encode("aaabbc") == "a3b2c1"
assert rle_encode("aabbcc") == "a2b2c2"
assert rle_encode("abcd") == "a1b1c1d1"

# Single character
assert rle_encode("a") == "a1"

# Spaces
assert rle_encode("aa bb") == "a2 2b2"

# Empty string
assert rle_encode("") == ""

# --- rle_decode ---

# Happy path — roundtrip consistency
assert rle_decode("a3b2c1") == "aaabbc"
assert rle_decode("a1b1c1d1") == "abcd"

# Multi-digit count
assert rle_decode("a12b3") == "a" * 12 + "bbb"

# Empty string
assert rle_decode("") == ""

# Error case — invalid format
import traceback
try:
    rle_decode("aab3")   # two letters before a digit
    assert False, "Expected ValueError"
except ValueError as e:
    assert str(e) == "Invalid encoded string"

# Roundtrip: encode then decode recovers the original
original = "aaabbbccddddeeee"
assert rle_decode(rle_encode(original)) == original
```

## Real-World Context

Run-length encoding is one of the oldest lossless compression schemes and is still in active use: it is embedded in fax protocols (ITU T.4), BMP and TIFF image formats, and the PCX raster format. A deeper version also underlies DEFLATE (used in PNG, ZIP, and HTTP gzip). Parsing alternating character/number tokens — as you do in `rle_decode` — is a micro-version of the lexer step that every compiler and config-file parser performs.

## Hints

<details>
<summary>Hint 1 — Conceptual nudge</summary>

For encoding, think of scanning the string and keeping track of the "current" character and how many times you have seen it in a row. Each time the character changes (or you reach the end), emit what you have accumulated and start a new run.

For decoding, you need to read two things at a time: one character, then one or more digit characters. Python's `str.isdigit()` method helps you tell them apart.

</details>

<details>
<summary>Hint 2 — More direct</summary>

**Encode:** Initialise `current_char = s[0]` and `count = 1`. Loop from index 1 onward. If `s[i] == current_char`, increment `count`; otherwise append `current_char + str(count)` to the result, reset `current_char = s[i]` and `count = 1`. After the loop, append the final pair.

**Decode:** Use an index `i`. While `i < len(s)`: read `s[i]` as the character (it must not be a digit — if it is, raise `ValueError`); advance `i`; collect consecutive digits into a number string; convert to `int`; append the character repeated that many times.

</details>

<details>
<summary>Hint 3 — Near-solution guidance</summary>

```python
# Decode skeleton
i = 0
result = []
while i < len(s):
    if s[i].isdigit():
        raise ValueError("Invalid encoded string")
    ch = s[i]
    i += 1
    num_str = ""
    while i < len(s) and s[i].isdigit():
        num_str += s[i]
        i += 1
    if not num_str:
        raise ValueError("Invalid encoded string")
    result.append(ch * int(num_str))
return "".join(result)
```

</details>

## Reference Solution

```python
def rle_encode(s: str) -> str:
    if not s:
        return ""

    result = []
    current_char = s[0]
    count = 1

    # Start from index 1 since we already "consumed" s[0] above.
    for i in range(1, len(s)):
        if s[i] == current_char:
            count += 1
        else:
            # Run ended: record the character and its count, then reset.
            result.append(current_char + str(count))
            current_char = s[i]
            count = 1

    # Append the final run (the loop exits without emitting it).
    result.append(current_char + str(count))
    return "".join(result)


def rle_decode(s: str) -> str:
    if not s:
        return ""

    result = []
    i = 0

    while i < len(s):
        # Each pair must start with a non-digit character.
        if s[i].isdigit():
            raise ValueError("Invalid encoded string")

        ch = s[i]
        i += 1

        # Collect all consecutive digit characters that follow.
        num_str = ""
        while i < len(s) and s[i].isdigit():
            num_str += s[i]
            i += 1

        # A character with no digits after it is malformed.
        if not num_str:
            raise ValueError("Invalid encoded string")

        # Repeat the character the specified number of times.
        result.append(ch * int(num_str))

    return "".join(result)
```

## Follow-Up Challenge

Write a function `rle_ratio(s: str) -> float` that computes the **compression ratio**: the length of the encoded string divided by the length of the original. A ratio below `1.0` means the encoding actually made the string longer (RLE is ineffective on strings with few repeated characters). Print a message indicating whether the encoding is beneficial.

```python
rle_ratio("aaaaaaa")   # 7 chars → "a7" (2 chars) → ratio ≈ 0.286  ✓ compressed
rle_ratio("abcdefg")   # 7 chars → "a1b1c1d1e1f1g1" (14 chars) → ratio = 2.0  ✗ expanded
```